In [3]:
#!/usr/bin/env python3
"""
Compare saved monopoles (or xi_s) from different runs.

Usage:
    python compare_saved_monopoles.py                         # interactive menu
    python compare_saved_monopoles.py --files f1.npz f2.npz   # plot specific files
    python compare_saved_monopoles.py --dirs /path1 /path2    # scan given directories
    python compare_saved_monopoles.py --output myplot.png     # save plot

The script looks for monopole files (name contains 'monopole') in the default data directories.
Each file should contain 's', 'xi0', and optionally 'label' (stored as string).
If 'label' is missing, the filename is used as label.
"""

import os
import glob
import argparse
import numpy as np
import matplotlib.pyplot as plt

# Default search paths – adjust to your setup
DEFAULT_SEARCH_PATHS = [
    "../data/monopoles/box_split_dist_fil",
    "../data/monopoles/box_split_Sigma_3",
    "../data/monopoles/box_split_dist_fil_Sigma_3",
    "../data/monopoles/box",  # fallback for no split runs
]

def load_monopole(filepath):
    """Load monopole data from npz, return (s, xi0, label)."""
    # Basic sanity checks
    if not os.path.isfile(filepath):
        raise FileNotFoundError(f"File not found: {filepath}")
    if not filepath.lower().endswith('.npz'):
        raise ValueError(f"File is not an .npz archive: {filepath}")
    try:
        data = np.load(filepath, allow_pickle=True)
    except Exception as e:
        raise ValueError(f"Failed to load {filepath}: {e}")

    # Extract required arrays
    s = data['s']
    xi0 = data['xi0']
    # Try to retrieve label; fallback to filename
    if 'label' in data:
        label = data['label'].item()
    else:
        label = os.path.basename(filepath).replace('.npz', '')
    return s, xi0, label

def find_all_monopole_files(search_paths=None):
    """Find all monopole .npz files in given directories (recursively)."""
    if search_paths is None:
        search_paths = DEFAULT_SEARCH_PATHS
    files = []
    for path in search_paths:
        if not os.path.exists(path):
            continue
        # Recursively search for .npz files containing 'monopole' in name
        for root, dirs, filenames in os.walk(path):
            for f in filenames:
                if f.endswith('.npz') and 'monopole' in f.lower():
                    files.append(os.path.join(root, f))
    return sorted(files)

def interactive_selection(files):
    """Present a numbered list of files for user to select."""
    print("Found monopole files:")
    valid_files = []
    for i, f in enumerate(files):
        # Try to read label without loading full data (optional, but we can load quickly)
        try:
            data = np.load(f, allow_pickle=True)
            label = data['label'].item() if 'label' in data else os.path.basename(f)
            valid_files.append((f, label))
        except Exception as e:
            print(f"  {i:3d}: {os.path.basename(f)} -> SKIPPED (could not read: {e})")
            continue

    if not valid_files:
        print("No readable monopole files found.")
        return []

    # Re‑number after filtering
    for idx, (f, label) in enumerate(valid_files):
        print(f"{idx:3d}: {label}")

    print("\nEnter numbers of files to plot (space‑separated), or 'all' to plot all, or 'quit':")
    choice = input("> ").strip()
    if choice.lower() == 'quit':
        return []
    if choice.lower() == 'all':
        return [f for f, _ in valid_files]
    try:
        indices = [int(x) for x in choice.split()]
        selected = [valid_files[i][0] for i in indices if 0 <= i < len(valid_files)]
        return selected
    except:
        print("Invalid selection.")
        return []

def plot_monopoles(selected_files, output_path=None, xlim=None, ylim=None):
    """Plot selected monopoles on a single figure."""
    fig, ax = plt.subplots(figsize=(8, 6))

    loaded_data = []
    for f in selected_files:
        try:
            s, xi0, label = load_monopole(f)
            loaded_data.append((s, xi0, label))
        except Exception as e:
            print(f"Warning: Skipping {f} – {e}")
            continue

    if not loaded_data:
        print("No valid monopole data to plot.")
        plt.close(fig)
        return

    for s, xi0, label in loaded_data:
        ax.plot(s, xi0 * s**2, marker='o', linestyle='-', label=label)

    ax.axhline(0, color='gray', linestyle='-', linewidth=1)
    ax.axvline(105, color='k', linestyle=':', label='BAO scale')
    ax.set_xlabel(r'$s\,[h^{-1}\mathrm{Mpc}]$')
    ax.set_ylabel(r'$s^{2}\xi_0(s)$')
    ax.set_title('Monopole Comparison')
    ax.legend(loc='upper right')
    if xlim:
        ax.set_xlim(xlim)
    if ylim:
        ax.set_ylim(ylim)
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=200, bbox_inches='tight')
        print(f"Saved plot to {output_path}")
    else:
        plt.show()
    plt.close()

def main():
    parser = argparse.ArgumentParser(description='Compare saved monopole files.')
    parser.add_argument('--files', nargs='+', help='List of specific .npz files to plot')
    parser.add_argument('--dirs', nargs='+', help='Directories to search for monopole files')
    parser.add_argument('--output', '-o', help='Output file for the plot (e.g., comparison.png)')
    parser.add_argument('--xlim', nargs=2, type=float, help='x‑axis limits (min max)')
    parser.add_argument('--ylim', nargs=2, type=float, help='y‑axis limits (min max)')
    args = parser.parse_args()

    # Determine files to plot
    if args.files:
        # Validate each file
        selected_files = []
        for f in args.files:
            if f.lower().endswith('.npz') and os.path.isfile(f):
                selected_files.append(f)
            else:
                print(f"Warning: Ignoring {f} – not a valid .npz file.")
        if not selected_files:
            print("No valid files specified.")
            return
    else:
        # Find all monopole files
        search_paths = args.dirs if args.dirs else None
        all_files = find_all_monopole_files(search_paths)
        if not all_files:
            print("No monopole files found. Check your search paths.")
            return
        # Interactive selection
        selected_files = interactive_selection(all_files)
        if not selected_files:
            print("No files selected.")
            return

    # Plot
    plot_monopoles(selected_files, args.output, args.xlim, args.ylim)

if __name__ == '__main__':
    main()

No valid files specified.
